# Adding emotion scores

Run after `text-classification.ipynb`. Score the sentences in each description and keep the highest score for each emotion. This is a simple ranking feature, not a measured description of how every reader will feel. The model downloads on first use.


In [1]:
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

from book_recommender.analysis import calculate_max_emotion_scores

books = pd.read_csv("books_with_categories.csv", dtype={"isbn13": str})
books[["title", "description"]].head()

c:\Python Projects\Book-Recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,title,description
0,Gilead,A NOVEL THAT READERS and critics have been eag...
1,Spider's Web,A new 'Christie for Christmas' -- a full-lengt...
2,Rage of angels,"A memorable, mesmerizing heroine Jennifer -- b..."
3,The Four Loves,Lewis' work on the nature of love divides love...
4,The Problem of Pain,"""In The Problem of Pain, C.S. Lewis, one of th..."


## Load the emotion model

CPU works on Windows and other platforms. `top_k=None` returns all emotion labels. Long inputs are truncated to the model's supported length.


In [2]:
classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=-1,
    truncation=True,
)

## Try a short example

Match scores by their label names, since the model can return labels in different orders.


In [3]:
example_predictions = classifier(["I love this!", "The ending was a surprise."])
calculate_max_emotion_scores(example_predictions)

{'anger': 0.004419783595949411,
 'disgust': 0.0016728027258068323,
 'fear': 0.0035983778070658445,
 'joy': 0.9771687984466553,
 'sadness': 0.005405383184552193,
 'surprise': 0.8896461725234985,
 'neutral': 0.016986170783638954}

## Score the book descriptions

Skip empty sentence fragments. Each book is processed once in this loop.


In [4]:
emotion_rows = []
for _, book in tqdm(books.iterrows(), total=len(books)):
    sentences = []
    for sentence in book["description"].split("."):
        sentence = sentence.strip()
        if sentence:
            sentences.append(sentence)

    if not sentences:
        raise ValueError(f"Book {book['isbn13']} has no description sentences")

    predictions = classifier(sentences)
    scores = calculate_max_emotion_scores(predictions)
    scores["isbn13"] = book["isbn13"]
    emotion_rows.append(scores)

emotions = pd.DataFrame(emotion_rows)
emotions.head()

100%|██████████| 5197/5197 [08:36<00:00, 10.06it/s]


,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.029641,0.338239,0.983973,0.949027,0.956065,0.729602,0.697846,9780002005883
1,0.594469,0.461991,0.935215,0.704422,0.051413,0.212222,0.891110,9780002261982
2,0.041301,0.024568,0.973285,0.767238,0.010860,0.009796,0.042176,9780006178736
3,0.325306,0.125763,0.436338,0.242209,0.043272,0.029084,0.732684,9780006280897
4,0.091732,0.197434,0.095043,0.041146,0.475881,0.074878,0.890048,9780006280934


## Join the scores and save

The original notebook mixed up some emotion labels when sorting model outputs. Regenerate these scores before using the updated app; old scores are not a valid comparison for this version.


In [5]:
books_with_emotions = books.merge(emotions, on="isbn13", validate="one_to_one")
books_with_emotions.to_csv("books_with_emotions.csv", index=False)
print(f"Saved {len(books_with_emotions)} books to books_with_emotions.csv")

Saved 5197 books to books_with_emotions.csv
